In [ ]:
import geopandas as gpd
import pandas as pd
import rioxarray
import matplotlib.pyplot as plt
import seaborn as sns
import os
import xarray as xr
from shapely.geometry import mapping


## Bringing GEE to Python 

Check out XEE


Xee is an Xarray backend for Google Earth Engine. Open ee.Image / ee.ImageCollection objects as lazy xarray.Datasets and analyze petabyte-scale Earth data with the scientific Python stack.


https://github.com/google/Xee

For further references checkout 

Google Earth Engine with Amirhossein Ahrari : https://www.youtube.com/@amirhosseinahrarigee

Check out his work on GEE ,'personal comment : he has solved most of your problems regarding  gee '

In [ ]:
import ee
import xee
ee.Authenticate()
ee.Initialize( opt_url='https://earthengine-highvolume.googleapis.com')

Running `ee.Authenticate()` for the first time will walk you through:

- **Opens a Google sign-in link** — printed in the notebook (or opened automatically in a browser)
- **Sign in** with the Google account registered for Earth Engine access
  (register at https://code.earthengine.google.com/register if not done yet)
- **Select your Google Cloud Project** — the one linked to your Earth Engine access
- **Approve access** — generates a verification code on the page
- **Paste the verification code** back into the notebook prompt:
  
- **Credentials are cached locally** (`~/.config/earthengine/credentials`) —
  future sessions on this machine won't need to re-authenticate

### After authentication, initialize with:
```python
ee.Initialize(
    project='your-gcp-project-id',
    opt_url='https://earthengine-highvolume.googleapis.com'
)
```

**Note:** `project` is required — use your registered Google Cloud Project ID,
not your email or account name.

In [ ]:
url = "https://geodata.ucdavis.edu/gadm/gadm4.1/shp/gadm41_IND_shp.zip"

india = gpd.read_file(url, layer="gadm41_IND_1")

# Select Kerala
kerala = india[india["NAME_1"] == "Kerala"]

print(kerala)

# Plot
ax = kerala.plot(figsize=(6, 8), edgecolor='black', facecolor='lightgreen')
ax.set_title("Kerala")

In [ ]:
bbox=kerala.bounds.values
bbox = bbox[0].tolist()
bbox

# Loading data from GEE

Data Details
https://developers.google.com/earth-engine/datasets/catalog/NASA_GPM_L3_IMERG_V07

In [ ]:
# date Fromat YYYY-MM-DD
time_start = ee.Date('2000-01-01')
time_end = ee.Date('2020-12-31')

# Number of days in range
time_dif = time_end.difference(time_start, 'day').round()

# Build list of daily steps
time_list = ee.List.sequence(0, ee.Number(time_dif).subtract(1), 1) \
    .map(lambda x: time_start.advance(x, 'day'))
#Imerge data 
col = ee.ImageCollection('NASA/GPM_L3/IMERG_V07').filterDate(time_start, time_end).select(["precipitation"])

# Function for daily aggregation
def daily(date):
    start_date = ee.Date(date)
    end_date = start_date.advance(1, 'day')
    img = col.filterDate(start_date, end_date).sum() \
             .set('system:time_start', start_date.millis())
    return img

# Map daily function over the date list
daily_col = ee.ImageCollection(time_list.map(daily))

# Bounding box (India example)
ds = xr.open_dataset(daily_col, engine = 'ee', crs = 'EPSG:4326', scale = 0.1, geometry = bbox)
# XEE gies mirror output need to transpse 
ds = ds.transpose('time', 'lat', 'lon')

# setting Spatial Dimentions

In [ ]:
ds = ds.rio.set_spatial_dims(x_dim='lon', y_dim='lat', inplace=True)
ds = ds.rio.write_crs('EPSG:4326', inplace=True)

print("CRS:", ds.rio.crs)
print("Resolution:", ds.rio.resolution())
#IMERG is 10 km .1 degree

In [ ]:
print ("Start date =",ds["precipitation"].time[0].dt.strftime('%Y-%m-%d').item())
print ("End date =",ds["precipitation"].time[-1].dt.strftime('%Y-%m-%d').item())

start_year = int(ds["precipitation"].time[0].dt.strftime('%Y').item())
end_year = int(ds["precipitation"].time[-1].dt.strftime('%Y').item())
year_diff = end_year - start_year
print('Numbner of years=',year_diff)

In [ ]:
f, ax = plt.subplots(figsize=(10, 4))#plotting overlayed data
ds["precipitation"][0].plot()

kerala.plot(ax=ax,
                 alpha=1)
plt.title("Overlay")



In [ ]:
ds = ds.rename({'lon': 'x', 'lat': 'y'})

# also make sure rioxarray actually knows the CRS —
# xee sets a 'crs' attribute, but rioxarray needs it written properly
ds = ds.rio.write_crs('EPSG:4326', inplace=True)


In [ ]:
# setting CRS 
ds["precipitation"]=ds["precipitation"].rio.write_crs(kerala.crs)
# Clipping 
ds["precipitation"]= ds["precipitation"].rio.clip(kerala.geometry.apply(mapping),
                                      kerala.crs,all_touched=True)# clipping data

In [ ]:
f, ax = plt.subplots(figsize=(10, 4))#plotting overlayed data
ds["precipitation"][0].plot()

kerala.plot(ax=ax,
                 alpha=.8)
plt.title("Overlay")

In [ ]:
kerala_rain=ds["precipitation"]

In [ ]:
#Spatial Average 
spat_avg=kerala_rain.mean(dim=["y","x"])

In [ ]:
f, ax = plt.subplots(figsize=(15, 4))
spat_avg.plot()
plt.title(f"Average daily Rainfall for Kerala: {start_year}-{end_year}")

In [ ]:
spat_avg

In [ ]:
total_annual_rainfall=spat_avg.groupby('time.year').sum()
total_annual_rainfall

In [ ]:
from matplotlib.ticker import MaxNLocator

fig, ax = plt.subplots(figsize=(15, 4))

total_annual_rainfall.plot(ax=ax)

ax.xaxis.set_major_locator(MaxNLocator(integer=True))

plt.title(f"Total annual Rainfall for Kerala: {start_year}-{end_year}")
plt.show()

## Save As .nc  in Local

In [ ]:
ds_out = ds.transpose('time', 'y', 'x').rename({'x': 'lon', 'y': 'lat'})
for var in ds_out.data_vars:
    ds_out[var].attrs = {k: v for k, v in ds_out[var].attrs.items()
                          if isinstance(v, (str, int, float))}

# 3. Compression encoding (keeps file size manageable for a 20-year daily series)
encoding = {var: {'zlib': True, 'complevel': 4} for var in ds_out.data_vars}

# 4. Write to disk
output_path = 'kerala_gpm_imerg_daily_2000_2025.nc'
ds_out.to_netcdf(output_path, encoding=encoding)

print(f"Saved to {output_path}, size: {os.path.getsize(output_path) / 1e6:.1f} MB")

In [ ]:
data=xr.open_dataset('kerala_gpm_imerg_daily_2000_2025.nc')

data
f, ax = plt.subplots(figsize=(10, 4))#plotting overlayed data
data["precipitation"][0].plot()

kerala.plot(ax=ax,
                 alpha=.8)
plt.title("Overlay")
data.close()

In [ ]:
print("CRS:",data.rio.crs)
print("Resolution:",data.rio.resolution())